# CURC VIIRS SNPP Workflow Planning

This notebook illustrates the CURC-specific workflow layer as a thin interactive front end over `workflows/curc`.

The intended user model is:
- choose a sensor, platform, tile, and water year,
- inspect discovered inputs and planned workflow steps,
- run one step at a time or eventually submit distributed Slurm jobs.

For this example, the source reflectance tree is the real SNPP `VNP09GA` archive on `/pl/active/rittger_ops/vnp09ga.002`.

In [5]:
from __future__ import annotations

from dataclasses import asdict
import json
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "spires").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"repo_root={REPO_ROOT}")

from spires.sensors.registry import list_supported_sensor_platforms
from workflows.curc.config import CurcWorkflowConfig
from workflows.curc.discovery import discover_viirs_snpp_water_year_reflectance_files
from workflows.curc.paths import ancillary_dir, log_root, output_date_root, reflectance_dir, r0_dir
from workflows.curc.runner import plan_viirs_snpp_steps


repo_root=/projects/ropa5718/SpiPy


## Configuration

The notebook should not define separate workflow logic, but it should expose the user-editable parameters clearly. This cell mirrors what a future external config file will contain.

In [6]:
supported = list_supported_sensor_platforms()
print(json.dumps(supported, indent=2))
print()

# User-editable workflow parameters.
SCRATCH_ROOT = Path("/scratch/alpine/ropa5718/spipy")
INPUT_SOURCE_ROOT = Path("/pl/active/rittger_ops/vnp09ga.002")
SENSOR = "viirs"
PLATFORMS = ("snpp",)
TILES = ("h09v05",)
WATER_YEARS = (2024,)
DATES = ()
DATE_GLOB = "*"
DRY_RUN = True

config = CurcWorkflowConfig(
    scratch_root=SCRATCH_ROOT,
    input_source_root=INPUT_SOURCE_ROOT,
    sensor=SENSOR,
    platforms=PLATFORMS,
    tiles=TILES,
    years=(),
    water_years=WATER_YEARS,
    dates=DATES,
    date_glob=DATE_GLOB,
    dry_run=DRY_RUN,
).canonicalized()

config


{
  "modis": [
    "terra",
    "aqua"
  ],
  "viirs": [
    "snpp",
    "noaa20",
    "noaa21"
  ]
}



CurcWorkflowConfig(scratch_root=PosixPath('/scratch/alpine/ropa5718/spipy'), input_source_root=PosixPath('/pl/active/rittger_ops/vnp09ga.002'), sensor='viirs', platforms=('snpp',), tiles=('h09v05',), years=(), water_years=(2024,), dates=(), date_glob='*', dry_run=True)

## Scratch Layout

The CURC workflow keeps shared staged inputs under `/scratch/alpine/.../input` and dated inversion outputs under `/scratch/alpine/.../output`.

In [7]:
platform = config.platforms[0]
tile = config.tiles[0]
water_year = config.water_years[0]

print("reflectance:", reflectance_dir(config, platform) / tile / str(water_year))
print("ancillary:", ancillary_dir(config, platform) / tile)
print("r0:", r0_dir(config, platform) / tile / str(water_year))
print("output_date:", output_date_root(config, platform, tile, "2024-04-03"))
print("logs:", log_root(config) / config.sensor / platform / tile / str(water_year))


reflectance: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h09v05/2024
ancillary: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h09v05
r0: /scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h09v05/2024
output_date: /scratch/alpine/ropa5718/spipy/output/viirs/snpp/h09v05/2024-04-03
logs: /scratch/alpine/ropa5718/spipy/logs/viirs/snpp/h09v05/2024


## Full Water-Year Discovery

This discovery step is the first concrete replacement for the opaque batch bash scripts. It gives a direct view of which source files are available for one tile and one water year.

In [8]:
reflectance_files = discover_viirs_snpp_water_year_reflectance_files(
    config,
    tile=tile,
    water_year=water_year,
)

print("water_year_file_count", len(reflectance_files))
print("first_file", reflectance_files[0] if reflectance_files else None)
print("last_file", reflectance_files[-1] if reflectance_files else None)


water_year_file_count 348
first_file /pl/active/rittger_ops/vnp09ga.002/input/h09v05/2023/VNP09GA.A2023274.h09v05.002.2023276013008.h5
last_file /pl/active/rittger_ops/vnp09ga.002/input/h09v05/2024/VNP09GA.A2024274.h09v05.002.2024275232423.h5


## Full Water-Year Step Plan

The workflow should be step-oriented rather than all-or-nothing. For a full water year, the planner expands the request into the major steps that will later become executable or submitted jobs.

In [9]:
full_water_year_steps = plan_viirs_snpp_steps(
    config,
    tile=tile,
    water_year=water_year,
)

for step in full_water_year_steps:
    summary = {
        "step": step.step,
        "date_count": step.date_count,
        "destination_path": step.destination_path,
        "r0_year": step.r0_year,
    }
    print(json.dumps(summary, indent=2))


{
  "step": "stage_reflectance",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h09v05/2024",
  "r0_year": null
}
{
  "step": "stage_ancillary",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h09v05",
  "r0_year": null
}
{
  "step": "build_r0",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/r0/h09v05/2024",
  "r0_year": 2024
}
{
  "step": "run_inversion",
  "date_count": 348,
  "destination_path": "/scratch/alpine/ropa5718/spipy/output/viirs/snpp/h09v05",
  "r0_year": 2024
}


## Single-Date Rerun Plan

A key design goal is that a user can rerun a single date without relaunching the entire water year. This is the concrete planning path for that case.

In [10]:
single_date = "2024-04-03"

single_date_steps = plan_viirs_snpp_steps(
    config,
    tile=tile,
    water_year=water_year,
    target_dates=(single_date,),
)

for step in single_date_steps:
    print(json.dumps(asdict(step), indent=2))


{
  "step": "stage_reflectance",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h09v05",
  "water_year": 2024,
  "date_count": 1,
  "dates": [
    "2024-04-03"
  ],
  "source_paths": [
    "/pl/active/rittger_ops/vnp09ga.002/input/h09v05/2024/VNP09GA.A2024094.h09v05.002.2024095105100.h5"
  ],
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/reflectance/h09v05/2024",
  "notes": [
    "Copy or rsync the discovered VNP09GA files from /pl to /scratch/alpine.",
    "This step can target a full water year or a single acquisition date."
  ],
  "r0_year": null
}
{
  "step": "stage_ancillary",
  "sensor": "viirs",
  "platform": "snpp",
  "tile": "h09v05",
  "water_year": 2024,
  "date_count": 1,
  "dates": [
    "2024-04-03"
  ],
  "source_paths": [],
  "destination_path": "/scratch/alpine/ropa5718/spipy/input/viirs/snpp/ancillary/h09v05",
  "notes": [
    "Ensure static ancillary inputs such as canopy, terrain, landcover, and water masks are staged.",
    "This is 

## Development Direction

The next implementation steps are not notebook-specific. They belong in `workflows/curc` and should later be callable from notebooks, scripts, and Slurm entrypoints:
- execute `stage_reflectance` as an explicit rsync/copy operation,
- execute `stage_ancillary` with clear checks for static tile inputs,
- implement summer-only source selection for `build_r0`,
- run inversion per date or per chunk while preserving the same step boundaries.